# Lab 1.4 &mdash; One Agent or Three: Measuring the Coordination Tax

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 1 &middot; Module 1 &mdash; Agents vs. Multi-Agent Systems**

### What you'll do
- Build a supervisor that routes work to specialist workers
- Instrument both designs -- steps, estimated tokens, wall time
- Score single-agent against multi-agent on one eval set
- Count the failure surface, and see why it does not grow linearly

> **How this lab works.** Fill every `BLANK`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, so your score never depends on a
> live endpoint. Cells marked **Run it for real** do call the sandbox model; if it is not
> reachable they print how to fix it instead of crashing.

> **Builds on Lab 1.3.** The claim 'we need multiple agents' is a hypothesis.
> This lab is how you test it. Workers here are deterministic stubs, so the numbers
> are reproducible and the comparison is about *architecture*, not model variance.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-1-04")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- graded cells still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 1 labs: payment exceptions on a small ledger.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

In [ ]:
# ------------------------------------------------- carried forward from Lab 1.2
# These are the tools you wrote in Lab 1.2. Nothing to fill in -- they are here so this
# notebook runs on its own. Note the docstrings: they name the case AND the boundary.

def lookup_payment(ref: str) -> str:
    """Return the ledger record for one payment reference such as 'PMT-1002'.

    Use when you need the status, amount, counterparty or reason code of a specific payment.
    Not for searching across payments.
    """
    record = LEDGER.get(ref)
    if record is None:
        return f"no payment found with reference {ref!r}"
    return json.dumps({"ref": ref, **record})


def policy_for(reason_code: str) -> str:
    """Return the operating policy for one failure reason code, e.g. 'LIMIT_BREACH'.

    Use after you know why a payment failed and need to know what to do about it.
    """
    return POLICY.get(reason_code, f"no policy on file for reason code {reason_code!r}")


TOOLS = {"lookup_payment": lookup_payment, "policy_for": policy_for}
print("carried forward:", ", ".join(TOOLS))

## Concept

Every agent you add costs 3&ndash;10&times; the tokens, adds latency, and opens a new class of failure.
Two things repay that tax, and only two: **specialisation** (the sub-tasks genuinely need
different tools) and **parallelism** (they genuinely run at the same time).

Anything else is tax paid for nothing. This lab produces the numbers that settle the argument.

## Section 1 &mdash; The supervisor's routing rule

The supervisor does one job: decide which worker gets the brief. Rule-based here, so the routing
is inspectable; Module 5 replaces the rule with a model.

In [ ]:
WORKERS = ("ledger", "policy", "critic")

def route(brief: dict) -> str:
    """Pick the worker for one brief. Returns a name from WORKERS.

    brief looks like {"ref": "PMT-1002", "needs": "status" | "policy" | "review"}
    """
    need = brief.get("needs")
    if need == "status":
        return "ledger"
    if need == "policy":
        return "policy"
    if need == "review":
        return "critic"
    return "ledger"                  # unknown need: start from the facts, never guess

In [ ]:
# --- Self-check: Section 1
check("a status brief goes to the ledger worker",
      lambda: route({"ref": "PMT-1002", "needs": "status"}) == "ledger")
check("a policy brief goes to the policy worker",
      lambda: route({"ref": "PMT-1002", "needs": "policy"}) == "policy")
check("a review brief goes to the critic",
      lambda: route({"ref": "PMT-1002", "needs": "review"}) == "critic")
check("an unknown need still routes to a real worker",
      lambda: route({"ref": "PMT-1002", "needs": "???"}) in WORKERS,
      "returning None would strand the brief")

## Section 2 &mdash; The meter

You cannot argue about the coordination tax without measuring it. A rough token estimate is fine
&mdash; what matters is that both designs are measured **the same way**.

In [ ]:
class Meter:
    """Counts what an architecture costs: steps, estimated tokens, wall time."""

    def __init__(self):
        self.steps = 0
        self.tokens = 0
        self.t0 = time.perf_counter()

    def record(self, prompt: str, response: str) -> None:
        """One model turn. Estimate ~4 characters per token -- crude, but applied equally."""
        self.steps += 1
        self.tokens += (len(prompt) + len(response)) // 4

    @property
    def seconds(self) -> float:
        return time.perf_counter() - self.t0

    def report(self) -> dict:
        return {"steps": self.steps, "tokens": self.tokens, "seconds": round(self.seconds, 4)}

In [ ]:
# --- Self-check: Section 2
def _metered():
    m = Meter()
    m.record("a" * 400, "b" * 400)
    return m

check("a turn advances the step count", lambda: _metered().report()["steps"] == 1)
check("both prompt and response are counted", lambda: _metered().report()["tokens"] == 200,
      "800 characters at ~4 chars/token is 200 -- count the prompt as well as the response")
check("two turns accumulate",
      lambda: (lambda m: (m.record("x" * 40, "y" * 40), m.report()["steps"])[1])(_metered()) == 2)
check("wall time is reported", lambda: _metered().report()["seconds"] >= 0)

## Section 3 &mdash; Two architectures, one eval set

The single agent handles a brief in one pass. The supervisor decomposes it, dispatches to
workers, and aggregates. Both answer the same six briefs, and both are metered identically.

In [ ]:
EVAL_SET = [
    {"ref": "PMT-1002", "question": "why did it fail?",        "expect": "INSUFFICIENT_FUNDS"},
    {"ref": "PMT-1003", "question": "can we release it?",      "expect": "Treasury"},
    {"ref": "PMT-1004", "question": "what do we do?",          "expect": "R04"},
    {"ref": "PMT-1005", "question": "who decides?",            "expect": "Compliance"},
    {"ref": "PMT-1001", "question": "is there an exception?",  "expect": "settled"},
    {"ref": "PMT-9999", "question": "why did it fail?",        "expect": "no payment found"},
]

def _worker(name, ref):
    """A deterministic stand-in for a specialist agent."""
    if name == "ledger":
        return lookup_payment(ref)
    if name == "policy":
        rec = LEDGER.get(ref)
        return policy_for(rec["reason_code"]) if rec and rec["reason_code"] else "no reason code"
    return "reviewed: findings are consistent with the ledger record"

def run_single(case, meter):
    """One agent, one pass: it holds every tool itself."""
    facts = lookup_payment(case["ref"])
    rec = LEDGER.get(case["ref"])
    policy = policy_for(rec["reason_code"]) if rec and rec["reason_code"] else ""
    answer = f"{facts} {policy}"
    meter.record(case["question"] + facts, answer)
    return answer

def run_supervised(case, meter):
    """Supervisor + workers: each hop re-reads the context it was handed."""
    answer = ""
    for need in ("status", "policy", "review"):
        worker = route({"ref": case["ref"], "needs": need})
        out = _worker(worker, case["ref"])
        meter.record(case["question"] + answer, out)      # the handoff re-sends what came before
        answer = (answer + " " + out).strip()
    return answer

def pass_rate(runner):
    """Fraction of EVAL_SET whose expected string appears in the answer, plus the meter report."""
    meter = Meter()
    hits = 0
    for case in EVAL_SET:
        answer = runner(case, meter)
        if case["expect"].lower() in answer.lower():
            hits += 1
    return {"pass_rate": round(hits / len(EVAL_SET), 3), **meter.report()}

In [ ]:
# --- Self-check: Section 3
_measured = {}

def measured(which):
    """Run one architecture over the eval set, once and then cached.

    Called from inside check(), so an unfilled blank above raises NameError there and prints
    [TODO]. Measuring into a zero-filled default instead would report [FAIL] on three of these
    checks -- and a false [PASS] on the fourth, since 0 == 0 * 3.
    """
    if which not in _measured:
        _measured[which] = pass_rate(run_single if which == "single" else run_supervised)
    return _measured[which]

guard(lambda: print("single    :", measured("single")))
guard(lambda: print("supervised:", measured("supervised")))
guard(lambda: print(
    f"\ntoken ratio: {measured('supervised')['tokens'] / max(measured('single')['tokens'], 1):.1f}x   "
    f"step ratio: {measured('supervised')['steps'] / max(measured('single')['steps'], 1):.1f}x"))

check("both architectures answer every case",
      lambda: measured("single")["steps"] == len(EVAL_SET)
              and measured("supervised")["steps"] == len(EVAL_SET) * 3)
check("the single agent scores on the eval set", lambda: measured("single")["pass_rate"] >= 0.8,
      "check the pass condition -- the expected string should appear in the answer")
check("the supervised design costs strictly more tokens",
      lambda: measured("supervised")["tokens"] > measured("single")["tokens"],
      "three hops re-read the context; that is the tax")
check("the supervised design takes three times the steps",
      lambda: measured("supervised")["steps"] == measured("single")["steps"] * 3)

## Section 4 &mdash; The failure surface

Tokens and latency grow roughly **linearly** with the agent count. The number of handoff paths
does not &mdash; and that is what you debug at 02:00.

In [ ]:
def handoff_paths(n_agents: int) -> int:
    """Ordered handoff paths between n agents: every agent may hand to every other."""
    return n_agents * (n_agents - 1)

In [ ]:
# --- Self-check: Section 4
check("two agents give two paths", lambda: handoff_paths(2) == 2)
check("three agents give six", lambda: handoff_paths(3) == 6)
check("five agents give twenty", lambda: handoff_paths(5) == 20,
      "n(n-1) -- not n(n-1)/2, because a handoff has a direction")
check("one agent has nothing to hand off to", lambda: handoff_paths(1) == 0)

for n in (1, 2, 3, 4, 5):
    try:
        print(f"{n} agents -> {handoff_paths(n):>2} handoff paths, ~{n}x tokens")
    except NameError:
        print("(fill in handoff_paths above)")
        break

### Read the numbers

You now have the argument in a form nobody can wave away: the supervised design costs measurably
more for this eval set and scores no better, because the work needs neither different tools nor
parallelism &mdash; the stubs read the same ledger. That is the coordination tax paid for nothing.

Note what would change the verdict: give the workers genuinely different tools, or run them
concurrently, and the ratios move. Module 5 does exactly that.

In [ ]:
score()

## Your turn

1. Make the policy worker slow (`time.sleep(0.05)`) and run the three workers concurrently with
   `concurrent.futures.ThreadPoolExecutor`. At what worker latency does parallelism start to pay
   for the extra tokens?
2. Add a seventh eval case the single agent gets **wrong** and the supervisor gets right. What
   property does that case need? If you cannot construct one, that is itself the finding.